# BGL Prediction (Now) — 1-minute HRV with 5-minute context
Predict the *current* CGM glucose using the 1-minute HRV features + temporal context (no prediction horizon).

In [1]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import neurokit2 as nk
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
PARTICIPANT = '001'
BASE_PATH = f'data/bigidea/{PARTICIPANT}/'
print('Ready')

Ready


In [2]:
# Load data
ibi_raw = pd.read_csv(BASE_PATH + f'IBI_{PARTICIPANT}.csv')
cgm_raw = pd.read_csv(BASE_PATH + f'Dexcom_{PARTICIPANT}.csv')
print('IBI rows:', len(ibi_raw), 'CGM rows:', len(cgm_raw))

IBI rows: 266366 CGM rows: 2573


In [3]:
# Clean CGM
cgm = cgm_raw[cgm_raw['Event Type'] == 'EGV'].copy()
cgm = cgm.dropna(subset=['Timestamp (YYYY-MM-DDThh:mm:ss)', 'Glucose Value (mg/dL)'])
cgm['datetime'] = pd.to_datetime(cgm['Timestamp (YYYY-MM-DDThh:mm:ss)'])
cgm = cgm[['datetime', 'Glucose Value (mg/dL)']].rename(columns={'Glucose Value (mg/dL)': 'glucose'})
cgm = cgm.sort_values('datetime').reset_index(drop=True)
print('CGM cleaned rows:', len(cgm))

CGM cleaned rows: 2561


In [4]:
# Clean IBI
ibi = ibi_raw.copy()
ibi.columns = ibi.columns.str.strip().str.lower().str.replace(' ', '_')
ibi = ibi.rename(columns={ibi.columns[0]: 'datetime', ibi.columns[1]: 'ibi'})
ibi['datetime'] = pd.to_datetime(ibi['datetime'])
ibi = ibi.sort_values('datetime').reset_index(drop=True)
ibi = ibi[(ibi['ibi'] >= 0.3) & (ibi['ibi'] <= 1.5)].copy()
ibi['ibi_diff_pct'] = ibi['ibi'].pct_change().abs()
ibi = ibi[(ibi['ibi_diff_pct'].isna()) | (ibi['ibi_diff_pct'] <= 0.20)].copy()
ibi = ibi.drop(columns=['ibi_diff_pct']).reset_index(drop=True)
print('IBI cleaned rows:', len(ibi))

IBI cleaned rows: 258831


## Build 1-minute HRV frame (labelled by minute end)

In [ ]:
HRV_BASE_FEATURES = ['HR', 'SDNN', 'SDSD', 'RMSSD', 'PNNI_20', 'PNNI_50', 'HR_MAD', 'CVNN', 'CVSD']
def compute_minute_hrv(ibi_window_seconds, min_beats=8):
    result = {f: np.nan for f in HRV_BASE_FEATURES}
    ibi_arr = np.asarray(ibi_window_seconds, dtype=float)
    ibi_arr = ibi_arr[~np.isnan(ibi_arr)]
    if len(ibi_arr) < min_beats:
        return result
    rri_ms = ibi_arr * 1000.0
    peaks_ms = np.append([0], np.cumsum(rri_ms))
    try:
        hrv_df = nk.hrv_time(peaks_ms, sampling_rate=1000)
        row = hrv_df.iloc[0]
        mean_nn = row.get('HRV_MeanNN', np.nan)
        result['HR'] = 60000.0 / mean_nn if pd.notna(mean_nn) and mean_nn>0 else np.nan
        for k, col in [('SDNN','HRV_SDNN'), ('SDSD','HRV_SDSD'), ('RMSSD','HRV_RMSSD'), ('PNNI_20','HRV_pNN20'), ('PNNI_50','HRV_pNN50'), ('HR_MAD','HRV_MadNN'), ('CVNN','HRV_CVNN'), ('CVSD','HRV_CVSD')]:
            result[k] = row[col] if col in hrv_df.columns else np.nan
        return result
    except Exception:
        return result

def build_minute_hrv_frame(ibi_df):
    start = ibi_df['datetime'].min().floor('min') + pd.Timedelta(minutes=1)
    end = ibi_df['datetime'].max().ceil('min')
    minute_ends = pd.date_range(start=start, end=end, freq='1min')
    records = []
    for window_end in minute_ends:
        window_start = window_end - pd.Timedelta(minutes=1)
        mask = (ibi_df['datetime'] >= window_start) & (ibi_df['datetime'] < window_end)
        window_values = ibi_df.loc[mask, 'ibi'].values
        feats = compute_minute_hrv(window_values, min_beats=8)
        feats['datetime'] = window_end
        feats['n_beats'] = len(window_values)
        records.append(feats)
    minute_hrv = pd.DataFrame(records).sort_values('datetime').reset_index(drop=True)
    return minute_hrv

minute_hrv = build_minute_hrv_frame(ibi)
print('minute_hrv rows:', len(minute_hrv))
minute_hrv.head()

## Temporal features: rolling, lag diffs, time-of-day

In [ ]:
# rolling over 5 minutes (min_periods=1 to keep early rows)
for col in HRV_BASE_FEATURES:
    minute_hrv[f'{col}_roll_mean5'] = minute_hrv[col].rolling(window=5, min_periods=1).mean()
    minute_hrv[f'{col}_roll_std5']  = minute_hrv[col].rolling(window=5, min_periods=1).std()
# time of day
seconds_in_day = 24*3600
t = minute_hrv['datetime'].dt.hour*3600 + minute_hrv['datetime'].dt.minute*60 + minute_hrv['datetime'].dt.second
minute_hrv['time_sin'] = np.sin(2*np.pi*t/seconds_in_day)
minute_hrv['time_cos'] = np.cos(2*np.pi*t/seconds_in_day)
minute_hrv['hour'] = minute_hrv['datetime'].dt.hour
minute_hrv['is_night'] = ((minute_hrv['hour'] >= 22) | (minute_hrv['hour'] <= 6)).astype(int)
minute_hrv['is_dawn']  = ((minute_hrv['hour'] >= 4)  & (minute_hrv['hour'] <= 8)).astype(int)

# create lagged base HRV and lag diffs (1..4)
def add_lags_and_diffs(df, cols, n_lags=4):
    df = df.sort_values('datetime').reset_index(drop=True).copy()
    for col in cols:
        for lag in range(1, n_lags+1):
            df[f'{col}_lag{lag}'] = df[col].shift(lag)
            df[f'{col}_diff_lag{lag}'] = df[col] - df[f'{col}_lag{lag}']
    return df

minute_hrv = add_lags_and_diffs(minute_hrv, HRV_BASE_FEATURES, n_lags=4)
print('Added temporal features')
minute_hrv.head()

## Align minute-HRV to CGM (no horizon shift)

In [ ]:
# prepare feature list
feature_cols = []
for col in HRV_BASE_FEATURES:
    feature_cols.append(col)
    for lag in range(1,5):
        feature_cols.append(f'{col}_lag{lag}')
        feature_cols.append(f'{col}_diff_lag{lag}')
    feature_cols.append(f'{col}_roll_mean5')
    feature_cols.append(f'{col}_roll_std5')
feature_cols.extend(['time_sin','time_cos','hour','is_night','is_dawn'])
# align using merge_asof (backward)
hrv_for_merge = minute_hrv.sort_values('datetime').copy()
hrv_for_merge['hrv_minute'] = hrv_for_merge['datetime']
cgm_for_merge = cgm.sort_values('datetime').copy()
aligned = pd.merge_asof(cgm_for_merge, hrv_for_merge, on='datetime', direction='backward')
aligned = aligned.dropna(subset=feature_cols + ['glucose']).sort_values('datetime').reset_index(drop=True)
aligned['alignment_gap_sec'] = (aligned['datetime'] - aligned['hrv_minute']).dt.total_seconds()
print('Aligned rows:', len(aligned))
print(aligned[['datetime','hrv_minute','alignment_gap_sec']].head())

In [ ]:
# Train/validation split (chronological)
TEST_FRACTION = 0.20
split_idx = int(len(aligned) * (1 - TEST_FRACTION))
df_train = aligned.iloc[:split_idx].copy()
df_val = aligned.iloc[split_idx:].copy()
X_train = df_train[feature_cols]
y_train = df_train['glucose']
X_val = df_val[feature_cols]
y_val = df_val['glucose']
print('Train / Val rows:', len(X_train), len(X_val))

In [ ]:
# Train XGBoost
model = XGBRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=3, subsample=0.8,
    colsample_bytree=0.8, min_child_weight=10, reg_alpha=0.5, reg_lambda=1.0,
    eval_metric='rmse', early_stopping_rounds=30, random_state=RANDOM_STATE, n_jobs=-1, verbosity=0
)
model.fit(X_train, y_train, eval_set=[(X_train,y_train),(X_val,y_val)], verbose=50)
print('Trained')

In [ ]:
def evaluate(y_true, y_pred, label):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mard = np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)) / np.maximum(np.asarray(y_true), 1e-6)) * 100
    print(f'\n{label}')
    print(f'MAE  : {mae:.3f} mg/dL')
    print(f'RMSE : {rmse:.3f} mg/dL')
    print(f'R^2  : {r2:.4f}')
    print(f'MARD : {mard:.2f}%')
    return {'mae': mae, 'rmse': rmse, 'r2': r2, 'mard': mard}

y_pred_train = model.predict(X_train)
y_pred_val = model.predict(X_val)
train_metrics = evaluate(y_train, y_pred_train, f'TRAIN - Participant {PARTICIPANT}')
val_metrics = evaluate(y_val, y_pred_val, f'VALIDATION - Participant {PARTICIPANT}')

In [ ]:
# Feature importance
importance_df = pd.DataFrame({'feature': feature_cols, 'importance': model.feature_importances_}).sort_values('importance', ascending=False)
print(importance_df.head(20).to_string(index=False))
plt.figure(figsize=(8,10))
sns.barplot(data=importance_df.head(20), x='importance', y='feature')
plt.title('Top features')
plt.tight_layout()
plt.show()